# scOPE — LUAD (TCGA Lung Adenocarcinoma)

1. **Phase 1** — TCGA LUAD bulk RNA-seq + MC3 mutations.
2. **Phase 2** — Kim et al. 2020 (Nature Communications) LUAD scRNA-seq
   (GSE131907, ~58 000 cells across lung cancer subtypes + normal).

**Data notes:**
- Bulk: Xena HiSeqV2_PANCAN log2(norm_count+1). Primary (01) + normal (11).
- SC: GSE131907 — `GSE131907_Lung_Cancer_normalized_log2TPM_matrix.txt.gz`
  is provided as log2(TPM+1) — **no further normalisation applied**.
  Filtering to LUAD cells (cell annotation `tissue_type` == "LUAD" or similar).


## 1. Imports & paths

In [ ]:
import os
import subprocess

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import requests

from scope import BulkPipeline, SingleCellPipeline
from scope.visualization import (
    compute_umap,
    plot_mutation_probabilities,
    plot_scree,
    plot_mutation_heatmap,
)

mpl.rcParams['figure.dpi']    = 100
mpl.rcParams['savefig.dpi']   = 300
mpl.rcParams['figure.figsize'] = (10, 6)


In [ ]:
BASE_DIR   = "/home/groups/precepts/ashforda/scOPE_github_stuff/scOPE_overhaul/scOPE/data"
BULK_DIR   = os.path.join(BASE_DIR, "TCGA_LUAD")
SC_DIR     = os.path.join(BASE_DIR, "LUAD_scRNA")
MODELS_DIR = os.path.join(BASE_DIR, "..", "models", "LUAD")
CANCER_TAG = "LUAD"

for d in [BULK_DIR, SC_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

luad_expr_path = os.path.join(BULK_DIR, "TCGA_LUAD_HiSeqV2_PANCAN.tsv.gz")
mc3_path       = os.path.join(BULK_DIR, "mc3.v0.2.8.PUBLIC.xena.gz")
sc_tpm_path    = os.path.join(SC_DIR,   "GSE131907_log2TPM_matrix.txt.gz")
sc_meta_path   = os.path.join(SC_DIR,   "GSE131907_cell_annotation.txt.gz")


## 2. Download raw data

In [ ]:
XENA_BASE = "https://tcga.xenahubs.net/download"
if os.path.exists(luad_expr_path):
    print(f"  already present -- {os.path.basename(luad_expr_path)}")
else:
    print("  downloading TCGA LUAD expression ...")
    r = requests.get(f"{XENA_BASE}/TCGA.LUAD.sampleMap/HiSeqV2_PANCAN.gz",
                     stream=True, timeout=300)
    r.raise_for_status()
    with open(luad_expr_path, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print(f"  done -> {luad_expr_path}")


In [ ]:
# -- MC3 PanCanAtlas MAF -- all TCGA cancer types, one file (~200 MB) -------
# Check sibling directories before downloading to avoid re-fetching.
_mc3_alts = [
    os.path.join(BASE_DIR, sib, "mc3.v0.2.8.PUBLIC.xena.gz")
    for sib in ["TCGA_CRC", "TCGA_SKCM", "TCGA_BRCA", "TCGA_LUAD",
                "TCGA_GBM", "TCGA_PAAD", "TCGA_LIHC", "TCGA_LAML"]
]
for _alt in _mc3_alts:
    if not os.path.exists(mc3_path) and os.path.exists(_alt):
        import shutil; shutil.copy(_alt, mc3_path)
        print(f"  copied MC3 from {os.path.dirname(_alt)}")
        break

if not os.path.exists(mc3_path):
    print("Downloading MC3 MAF (~200 MB) ...")
    r = requests.get(
        "https://pancanatlas.xenahubs.net/download/mc3.v0.2.8.PUBLIC.xena.gz",
        stream=True, timeout=600,
    )
    r.raise_for_status()
    with open(mc3_path, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print(f"  done -> {mc3_path}")
else:
    print(f"  already present -- {os.path.basename(mc3_path)}")


In [ ]:
def _download_ncbi(url: str, dest: str, retries: int = 3) -> None:
    """Download from NCBI GEO via wget, bypassing SSL EOF on HPC nodes."""
    cmd = [
        "wget", "--quiet", "--show-progress",
        f"--tries={retries}", "--timeout=120",
        "--no-check-certificate",
        "-O", dest, url,
    ]
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        if os.path.exists(dest):
            os.remove(dest)
        raise RuntimeError(f"wget failed (rc={result.returncode}) for {url}")

GEO_HTTP = "https://www.ncbi.nlm.nih.gov/geo/download"
sc_downloads = {
    sc_tpm_path  : (f"{GEO_HTTP}/?acc=GSE131907&format=file"
                    "&file=GSE131907%5FLung%5FCancer%5Fnormalized%5Flog2TPM%5Fmatrix%2Etxt%2Egz"),
    sc_meta_path : (f"{GEO_HTTP}/?acc=GSE131907&format=file"
                    "&file=GSE131907%5FLung%5FCancer%5Fcell%5Fannotation%2Etxt%2Egz"),
}
for dest, url in sc_downloads.items():
    if os.path.exists(dest):
        print(f"  already present -- {os.path.basename(dest)}")
    else:
        print(f"  downloading {os.path.basename(dest)} ...  (may take several minutes)")
        _download_ncbi(url, dest)
        print(f"  done -> {dest}")


## 3. Load & prepare bulk RNA-seq

In [ ]:
def load_xena_expression(path, cohort, type_map=None):
    """Load Xena HiSeqV2_PANCAN -> samples x genes AnnData."""
    if type_map is None:
        type_map = {"01": "tumor", "11": "normal"}
    df = pd.read_csv(path, sep="\t", index_col=0).T
    keep = df.index.str[13:15].isin(type_map)
    df   = df[keep]
    sample_type = df.index.str[13:15].map(type_map)
    adata = ad.AnnData(
        X   = df.values.astype(np.float32),
        obs = pd.DataFrame({"cohort": cohort, "sample_type": sample_type.values},
                           index=df.index),
        var = pd.DataFrame(index=df.columns),
    )
    for st, label in type_map.items():
        n = (sample_type == label).sum()
        print(f"  {cohort} {label} : {n}")
    return adata


In [ ]:
adata_bulk = load_xena_expression(luad_expr_path, "LUAD")
adata_bulk.var_names_make_unique()
print(f"Bulk : {adata_bulk.n_obs} x {adata_bulk.n_vars}")
X = adata_bulk.X
print(f"NaN : {np.isnan(X).sum()} | Min : {np.nanmin(X):.2f} | Max : {np.nanmax(X):.2f}")


## 4. Build mutation label matrix

In [ ]:
LUAD_GENES = [
    "KRAS",     # ~30% -- codons 12/13; defines targeted-therapy cohort
    "EGFR",     # ~15% -- exon 19 del / L858R; mutually exclusive with KRAS
    "TP53",     # ~50%
    "STK11",    # LKB1 -- co-mutation with KRAS; immunotherapy resistance
    "KEAP1",    # NRF2 pathway; co-mutation with KRAS
    "NF1",
    "BRAF",     # ~5% -- mostly non-V600E in LUAD
    "PIK3CA",
    "MET",      # exon 14 skipping
    "ERBB2",    # exon 20 insertions
    "U2AF1",
    "RBM10",
    "CDKN2A",
    "RB1",
    "SMARCA4",
    "ARID1A",
    "SETD2",
    "MGA",
    "ATM",
    "PTPRD",
]
LUAD_GENES = list(dict.fromkeys(LUAD_GENES))
print(f"Targeting {len(LUAD_GENES)} LUAD driver genes")


In [ ]:
KEEP_CLASSES = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins",
    "Splice_Site", "Translation_Start_Site", "Nonstop_Mutation",
}


In [ ]:
# -- Filter MC3 to this cancer by matching against bulk 15-char barcodes --
mc3 = pd.read_csv(mc3_path, sep="\t", low_memory=False)
mc3 = mc3.rename(columns={
    "sample": "Tumor_Sample_Barcode",
    "effect": "Variant_Classification",
    "gene":   "Hugo_Symbol",
})
bulk_15_set   = set(adata_bulk.obs_names.str[:15])
mc3["sample_id"] = mc3["Tumor_Sample_Barcode"].str[:15]
maf_cancer    = mc3[mc3["sample_id"].isin(bulk_15_set)].copy()
print(f"Cancer rows (pre-filter) : {len(maf_cancer):,}   "
      f"samples: {maf_cancer['sample_id'].nunique()}")

maf_cancer = maf_cancer[maf_cancer["Variant_Classification"].isin(KEEP_CLASSES)]
maf_all    = maf_cancer[["sample_id", "Hugo_Symbol"]].dropna()
print(f"After coding filter      : {len(maf_all):,} variants   "
      f"{maf_all['sample_id'].nunique()} samples")


In [ ]:
mut_matrix = (
    maf_all[["sample_id", "Hugo_Symbol"]].drop_duplicates()
    .assign(mutated=1)
    .pivot_table(index="sample_id", columns="Hugo_Symbol",
                 values="mutated", fill_value=0)
)
mut_matrix.columns.name = None; mut_matrix.index.name = None
genes_present = [g for g in LUAD_GENES if g in mut_matrix.columns]
mutation_labels_raw = mut_matrix[genes_present]
is_normal = adata_bulk.obs["sample_type"] == "normal"
has_mc3   = adata_bulk.obs_names.str[:15].isin(mutation_labels_raw.index)
keep_mask = has_mc3 | is_normal.values
adata_bulk = adata_bulk[keep_mask].copy()
adata_bulk.obs["barcode_15"] = adata_bulk.obs_names.str[:15]
mut_re = mutation_labels_raw.copy()
mut_re.index = mut_re.index.str[:15]
mut_re = mut_re[~mut_re.index.duplicated(keep="first")]
mutation_labels = mut_re.reindex(adata_bulk.obs["barcode_15"]).fillna(0).astype(int)
mutation_labels.index = adata_bulk.obs_names
print(f"Mutation matrix : {mutation_labels.shape}")
tmask = adata_bulk.obs["sample_type"] != "normal"
print(mutation_labels[tmask].sum().sort_values(ascending=False).head(15))


## 5. Load single-cell data — Kim et al. 2020 (GSE131907)

In [ ]:
import pandas as pd
import numpy as np
import anndata as ad
import gzip

# --- 1. Read only the header to get column names, then subset cols on read ---
print("Scanning header ...")
meta = pd.read_csv(sc_meta_path, sep="\t", index_col=0)
print(f"Metadata : {meta.shape}  cols: {meta.columns.tolist()}")

#with open(sc_tpm_path) as f:
#    header_cols = f.readline().rstrip("\n").split("\t")
with gzip.open(sc_tpm_path, "rt") as f:
    header_cols = f.readline().rstrip("\n").split("\t")

# header_cols[0] is the index col label, rest are cell IDs
all_cells = header_cols[1:]
common_cells = list(meta.index.intersection(all_cells))
print(f"Common cells: {len(common_cells)}")


In [ ]:
# --- 2. Read only the needed columns, directly as float32 ---
print("Loading TPM matrix (subset cols) ...")
tpm_df = pd.read_csv(
    sc_tpm_path,
    sep="\t",
    index_col=0,
    usecols=[header_cols[0]] + common_cells,
    dtype=np.float32,
    compression="gzip",
)
print(f"Loaded: {tpm_df.shape}")


In [ ]:
# --- 3. Transpose in numpy (much faster than pandas .T) ---
print("Building AnnData ...")
X = tpm_df.values.T  # already float32, no copy needed
adata_sc_all = ad.AnnData(
    X   = X,
    obs = meta.loc[common_cells],
    var = pd.DataFrame(index=tpm_df.index),
)
adata_sc_all.var_names_make_unique()
print(f"SC loaded : {adata_sc_all.n_obs} cells x {adata_sc_all.n_vars} genes")


In [ ]:
# -- Filter to LUAD cells --------------------------------------------------
# The Kim 2020 dataset includes multiple lung cancer subtypes; restrict to LUAD.
# Inspect the annotation column above and adjust the filter string if needed.
if ct_col:
    luad_mask = adata_sc_all.obs[ct_col].str.upper().str.contains("LUAD", na=False)
    adata_sc  = adata_sc_all[luad_mask].copy()
    print(f"LUAD cells : {adata_sc.n_obs} / {adata_sc_all.n_obs}")
    if adata_sc.n_obs < 1000:
        print("WARNING: few LUAD cells -- check ct_col filter above.")
        print("Available values:", adata_sc_all.obs[ct_col].value_counts().head(10))
else:
    print("No cell-type column found -- using all cells.")
    adata_sc = adata_sc_all.copy()

# QC (data already log2(TPM+1) -- no renormalisation)
sc.pp.filter_genes(adata_sc, min_cells=5)
sc.pp.filter_cells(adata_sc, min_genes=100)
print(f"After QC : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")
print(f"Value range: {adata_sc.X.min():.2f} -- {adata_sc.X.max():.2f}")
print("(log2(TPM+1) -- no further normalisation applied.)")


## 6. Gene overlap & sanity checks

In [ ]:
overlap = set(adata_bulk.var_names)&set(adata_sc.var_names)
print(f'Gene overlap : {len(overlap):,}')
mutation_labels.head()

## 7. Phase 1 — Bulk pipeline

In [ ]:
n_components = 100
print(f'n_components = {n_components}')

In [ ]:
bulk_pipe = BulkPipeline(
    norm_method         = "none",
    log1p               = False,
    center              = True,
    scale               = True,
    decomposition       = "svd",
    n_components        = n_components,
    classifier          = "logistic",
    min_positive_frac   = 0.0001,
    classifier_kwargs   = {
        "C"        : 1.0,
        "solver"   : "saga",
        "max_iter" : 100000,
    },
)

bulk_pipe.fit(adata_bulk, mutation_labels, cv=10)
bulk_pipe.save(os.path.join(MODELS_DIR, f"{CANCER_TAG}_bulk_pipeline.pkl"))
print("Phase 1 complete -- model saved.")


In [ ]:
scree = bulk_pipe.decomposer_.scree_data()
fig, ax = plot_scree(scree, max_components=n_components); plt.tight_layout()
fig.savefig(os.path.join(MODELS_DIR, f"{CANCER_TAG}_scree.pdf"), bbox_inches="tight"); plt.show()
print(bulk_pipe.cv_results_.groupby("mutation")[["auroc","auprc"]].mean().round(3))


## 8. Phase 2 — Single-cell pipeline

In [ ]:
adata_bulk_pp = bulk_pipe.preprocessor_.transform(adata_bulk)

sc_pipe = SingleCellPipeline(
    bulk_pipeline    = bulk_pipe,
    alignment_method = "moment_matching",
)
sc_pipe.fit(adata_bulk_pp, adata_sc)
adata_sc = sc_pipe.transform(adata_sc)

mut_prob_cols = [c for c in adata_sc.obs.columns if c.startswith("mutation_prob_")]
print(f"Inferred {len(mut_prob_cols)} mutation probability columns")


## 9. Visualisation

In [ ]:
# -- UMAP on SVD latent embedding -----------------------------------------
adata_sc = compute_umap(adata_sc, obsm_key="X_svd")

top_muts = mutation_labels.sum().sort_values(ascending=False).index.tolist()
prob_cols = [f"mutation_prob_{g}" for g in top_muts
             if f"mutation_prob_{g}" in adata_sc.obs.columns]

# Cell type + top mutation UMAPs
ct_col = next((c for c in adata_sc.obs.columns
               if "type" in c.lower() or "celltype" in c.lower()
               or "cluster" in c.lower()), None)
if ct_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sc.pl.umap(adata_sc, color=ct_col, ax=axes[0], show=False, title="Cell type")
    sc.pl.umap(adata_sc, color=prob_cols[0] if prob_cols else ct_col,
               ax=axes[1], show=False, title=f"P({top_muts[0]} mut)")
    plt.tight_layout()
    fig.savefig(os.path.join(MODELS_DIR, f"{CANCER_TAG}_umap_overview.pdf"),
                bbox_inches="tight")
    plt.show()

# All mutation probability UMAPs
vmaxes = {c: max(float(np.percentile(adata_sc.obs[c].values, 99)), 0.0005)
          for c in prob_cols}
ncols = 3
nrows = int(np.ceil(len(prob_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()
for i, col in enumerate(prob_cols):
    gene = col.replace("mutation_prob_", "")
    sc.pl.umap(adata_sc, color=col, ax=axes[i], show=False,
               title=f"P({gene} mut)", vmin=0, vmax=vmaxes[col],
               cmap="RdBu_r", colorbar_loc="right", s=4)
for j in range(len(prob_cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(MODELS_DIR, f"{CANCER_TAG}_mutation_probs_umap.pdf"),
            bbox_inches="tight")
plt.show()

# Heatmap: mean probability per cluster
if ct_col:
    fig, ax = plot_mutation_heatmap(adata_sc, cluster_key=ct_col, mutations=top_muts)
    fig.savefig(os.path.join(MODELS_DIR, f"{CANCER_TAG}_heatmap_{ct_col}.pdf"),
                bbox_inches="tight")
    plt.show()


## 10. Save

In [ ]:
adata_sc.write_h5ad(os.path.join(MODELS_DIR, f"{CANCER_TAG}_sc_with_mutation_probs.h5ad"))
print(f"Saved h5ad -> {MODELS_DIR}")
